# 03 · Multi-Step Agents
### Practical LLM Engineering — Module 05: LLM Agents

> **What you'll learn**
> - The ReAct (Reason + Act) agent loop
> - Plan-and-Execute: decompose → execute → synthesise
> - Reflection agents: self-critique and retry
> - Agent trajectories: thought → action → observation
> - Budget enforcement: max steps, tokens, and time
> - Engineering: human-in-the-loop and safe termination

---


## 1. Overview

A **multi-step agent** breaks complex tasks into a loop of reasoning and action steps:

```
                    ┌──────────────────────────────────┐
                    │          ReAct Loop               │
                    │                                   │
  Task ──────────▶  │  Thought: "I need to find X"     │
                    │      ↓                            │
                    │  Action: search("X")              │
                    │      ↓                            │
                    │  Observation: [search result]     │
                    │      ↓                            │
                    │  Thought: "Now I can compute Y"   │
                    │      ↓                            │
                    │  Action: calculator("Y = ...")    │
                    │      ↓                            │
                    │  Final Answer: "Y = 42"           │
                    └──────────────────────────────────┘
```

### Agent architectures

| Architecture | Plans first? | Self-reflects? | Best for |
|---|---|---|---|
| **ReAct** | No | No | Simple multi-step tasks |
| **Plan-and-Execute** | Yes | No | Tasks with clear decomposition |
| **Reflexion** | No | Yes | Tasks needing iterative refinement |
| **LATS** | Yes | Yes | Complex open-ended tasks |

### Module position
```
01_function_calling   ✅
02_tool_use           ✅
03_multi_step_agents  ← you are here
04_agent_memory
```


## 2. Setup

In [ ]:
# !pip install numpy matplotlib -q
import os, re, json, time, math, random, io, contextlib
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Optional, Any
from collections import defaultdict
import math as _math

@dataclass
class LLMResponse:
    text: str; input_tokens: int; output_tokens: int; latency_ms: float
    @property
    def total_tokens(self): return self.input_tokens + self.output_tokens

class MockLLM:
    def __init__(self, seed=42): self.rng = random.Random(seed); self._n = 0
    def complete(self, system, user, tools=None, max_tokens=512, temperature=0.0):
        time.sleep(0.02); self._n += 1
        text = self._respond(system, user)
        return LLMResponse(text, len((system+user).split()), len(text.split()), 40.0)
    def _respond(self, system, user):
        u = user.lower()
        # Planner: return JSON subtasks
        if "json array" in u or "subtask" in u:
            return json.dumps([
                {"subtask": f"Search for information about the main topic", "rationale": "gather data"},
                {"subtask": f"Calculate or compute the numerical result", "rationale": "derive answer"},
            ])
        # Critic: return evaluation JSON
        if "evaluation" in u or "evaluate" in u or "score" in u:
            return json.dumps({"score": self.rng.uniform(6, 9), "issues": ["could be more detailed"],
                                "should_retry": self._n % 3 == 0})
        # ReAct: alternate thought/action/final
        if "thought:" in system.lower() or "react" in system.lower():
            if self._n % 3 == 0:
                return f"Thought: I now have enough information to answer.\nFinal Answer: The answer is based on careful analysis of {user[:40]}."
            tools_mentioned = re.findall(r"(calculator|search|python|read_file|write_file)", u)
            tool = tools_mentioned[0] if tools_mentioned else "calculator"
            return f'Thought: I need to use {tool} to proceed.\nAction: {tool} | {{"expression": "sqrt(144)", "query": "example", "code": "print(42)", "filename": "data.txt"}}'
        return f"Based on careful analysis: {user[:60]}... The answer follows from the provided context."
    def __call__(self, system, user, **kw): return self.complete(system, user, **kw)

llm = MockLLM()

# Minimal tool set
def _exec_py(code):
    buf, safe = io.StringIO(), {"print":print,"range":range,"len":len,"sum":sum,"abs":abs,
                                  "sorted":sorted,"list":list,"str":str,"int":int,"float":float,
                                  "min":min,"max":max,"math":_math}
    try:
        with contextlib.redirect_stdout(buf): exec(code, {"__builtins__": safe})
        return buf.getvalue().strip() or "[no output]"
    except Exception as e: return f"Error: {e}"

TOOLS = {
    "calculator": {"fn": lambda expression: eval(expression, {"__builtins__":{}},
                                                  {k:getattr(_math,k) for k in dir(_math) if not k.startswith("_")}),
                   "schema": {"name":"calculator","description":"Evaluate math expression.",
                               "input_schema":{"type":"object","properties":{"expression":{"type":"string"}},"required":["expression"]}}},
    "search":     {"fn": lambda query, max_results=3: {"results": [{"title": f"Result: {query}", "content": f"Detailed info about {query} including key concepts."}]},
                   "schema": {"name":"search","description":"Search for information.",
                               "input_schema":{"type":"object","properties":{"query":{"type":"string"}},"required":["query"]}}},
    "python":     {"fn": _exec_py,
                   "schema": {"name":"python","description":"Run Python code.",
                               "input_schema":{"type":"object","properties":{"code":{"type":"string"}},"required":["code"]}}},
    "write_file": {"fn": lambda filename, content: {"status":"written","filename":filename,"bytes":len(content)},
                   "schema": {"name":"write_file","description":"Write content to a file.",
                               "input_schema":{"type":"object","properties":{"filename":{"type":"string"},"content":{"type":"string"}},"required":["filename","content"]}}},
}

def call_tool(name, arguments):
    if name not in TOOLS: return {"error": f"Unknown tool: {name}"}
    try: return TOOLS[name]["fn"](**arguments)
    except Exception as e: return {"error": str(e)}

print(f"Tools: {list(TOOLS.keys())}")
print("✅ Setup complete")


## 3. ReAct: Reason + Act

**ReAct** (Yao et al., 2022) interleaves reasoning traces with actions. The model writes its thought before calling a tool, making the trajectory interpretable.

Prompt format:
```
Thought: <reasoning about what to do>
Action: <tool_name> | <json arguments>
Observation: [application inserts tool result]
... repeat ...
Final Answer: <complete answer to the task>
```


In [ ]:
@dataclass
class AgentStep:
    step_num:     int
    thought:      str = ""
    action:       Optional[str] = None
    action_args:  Optional[dict] = None
    observation:  Optional[str] = None
    is_final:     bool = False
    final_answer: Optional[str] = None
    latency_ms:   float = 0.0


class ReActAgent:
    SYSTEM = """You are a helpful agent that solves tasks step by step.
At each step output EXACTLY:
  Thought: <your reasoning>
  Action: <tool_name> | <json arguments>

When you have a complete answer:
  Thought: <final reasoning>
  Final Answer: <your complete answer>

Available tools:
{tool_list}"""

    def __init__(self, llm, tools, max_steps=8, verbose=True):
        self.llm, self.tools = llm, tools
        self.max_steps, self.verbose = max_steps, verbose

    def _tool_list(self):
        return "\n".join(f"  {t['schema']['name']}: {t['schema']['description']}"
                           for t in self.tools.values())

    def _parse(self, text):
        thought = (re.search(r"Thought:\s*(.*?)(?=Action:|Final Answer:|$)", text, re.DOTALL)
                   or type('',(),{'group':lambda s,n: ''})())
        thought = (thought.group(1) if hasattr(thought,'group') and callable(thought.group)
                   else "").strip()
        # Check for final answer
        m_fa = re.search(r"Final Answer:\s*(.*?)$", text, re.DOTALL)
        if m_fa: return thought, None, None, True, m_fa.group(1).strip()
        # Check for action
        m_a = re.search(r"Action:\s*(\w+)\s*\|\s*(\{.*?\})", text, re.DOTALL)
        if m_a:
            try:   args = json.loads(m_a.group(2))
            except: args = {}
            return thought, m_a.group(1).strip(), args, False, None
        # Fallback: try to detect tool names
        for name in self.tools:
            if name in text.lower():
                return thought, name, {}, False, None
        return thought, None, None, False, None

    def _trajectory(self, steps):
        parts = []
        for s in steps:
            if s.thought:      parts.append(f"Thought: {s.thought}")
            if s.action:       parts.append(f"Action: {s.action} | {json.dumps(s.action_args)}")
            if s.observation:  parts.append(f"Observation: {s.observation}")
            if s.is_final:     parts.append(f"Final Answer: {s.final_answer}")
        return "\n".join(parts)

    def _print(self, s):
        if s.thought:    print(f"  [{s.step_num}] Thought: {s.thought[:80]}")
        if s.action:     print(f"  [{s.step_num}] Action:  {s.action}({json.dumps(s.action_args)[:50]})")
        if s.observation:print(f"  [{s.step_num}] Obs:     {s.observation[:80]}")
        if s.is_final:   print(f"  [{s.step_num}] ✅ Answer: {s.final_answer[:100]}")
        print()

    def run(self, task):
        system    = self.SYSTEM.format(tool_list=self._tool_list())
        steps     = []
        total_tok = 0
        if self.verbose: print(f"Task: {task}\n")

        for n in range(1, self.max_steps+1):
            traj    = self._trajectory(steps)
            user    = f"Task: {task}\n\n{traj}" if traj else f"Task: {task}"
            t0      = time.perf_counter()
            resp    = self.llm(system, user, max_tokens=300)
            ms      = (time.perf_counter()-t0)*1000
            total_tok += resp.total_tokens

            thought, action, args, is_final, answer = self._parse(resp.text)
            step = AgentStep(n, thought, action, args, is_final=is_final,
                              final_answer=answer, latency_ms=ms)

            if is_final:
                steps.append(step)
                if self.verbose: self._print(step)
                break

            if action and action in self.tools:
                obs = call_tool(action, args or {})
                step.observation = json.dumps(obs)[:250]
            else:
                step.observation = "No valid action found."

            steps.append(step)
            if self.verbose: self._print(step)

        final = steps[-1].final_answer if steps and steps[-1].is_final else "Max steps reached."
        return {"task": task, "answer": final, "steps": steps,
                "n_steps": len(steps), "total_tokens": total_tok}


agent = ReActAgent(llm, TOOLS, max_steps=5, verbose=True)
result = agent.run("What is sqrt(2025), and what does that number squared equal?")
print(f"Steps used: {result['n_steps']}  Total tokens: {result['total_tokens']}")


## 4. Plan-and-Execute Agent

In [ ]:
class PlanExecuteAgent:
    """Phase 1: decompose task into subtasks. Phase 2: execute each. Phase 3: synthesise."""
    PLANNER = """You are a task planning assistant.
Break the given task into 2-4 independent subtasks.
Output ONLY a JSON array: [{"subtask": "...", "rationale": "..."}, ...]"""

    def __init__(self, llm, tools, max_steps_per=3):
        self.llm    = llm
        self.tools  = tools
        self.react  = ReActAgent(llm, tools, max_steps=max_steps_per, verbose=False)

    def plan(self, task):
        resp = self.llm(self.PLANNER, f"Task: {task}\n\nSubtasks (JSON):", max_tokens=300)
        try:
            raw  = re.sub(r"```(?:json)?\s*", "", resp.text).strip().rstrip("`")
            plan = json.loads(raw)
            if isinstance(plan, list): return [p for p in plan if "subtask" in p]
        except: pass
        return [{"subtask": task, "rationale": "Execute directly"}]

    def execute(self, task, verbose=True):
        plan = self.plan(task)
        if verbose:
            print(f"Plan ({len(plan)} subtasks):")
            for i, p in enumerate(plan, 1):
                print(f"  {i}. {p['subtask'][:65]}")
            print()

        results = []
        for i, p in enumerate(plan, 1):
            if verbose: print(f"Subtask {i}: {p['subtask'][:55]}")
            r = self.react.run(p["subtask"])
            results.append({"subtask": p["subtask"], "answer": r["answer"], "n_steps": r["n_steps"]})
            if verbose: print(f"  → {r['answer'][:80]}\n")

        ctx  = "\n".join(f"Subtask: {r['subtask']}\nResult: {r['answer']}" for r in results)
        resp = self.llm("Synthesise subtask results into one final answer.",
                         f"Task: {task}\n\nResults:\n{ctx}\n\nFinal answer:")
        return {"task": task, "plan": plan, "subtask_results": results,
                "final_answer": resp.text}


pe = PlanExecuteAgent(llm, TOOLS, max_steps_per=3)
result = pe.execute(
    "Research the HNSW algorithm, compute 16*32 connections for a graph, then save a summary to summary.txt",
    verbose=True)
print(f"Final answer: {result['final_answer'][:150]}")


## 5. Reflexion: Self-Critique and Retry

**Reflexion** (Shinn et al., 2023) adds a self-critique step: after generating an answer the agent scores its own output and retries if quality is insufficient.


In [ ]:
class ReflexionAgent:
    """Generates an answer, critiques it, and retries up to max_retries times."""
    CRITIC = """You are a critical evaluator. Rate the answer on completeness and accuracy.
Respond ONLY with JSON: {"score": 0-10, "issues": [...], "should_retry": bool}"""

    def __init__(self, llm, tools, max_retries=2, threshold=7.5):
        self.llm, self.tools  = llm, tools
        self.max_retries, self.threshold = max_retries, threshold
        self.react = ReActAgent(llm, tools, max_steps=4, verbose=False)

    def _critique(self, task, answer):
        resp = self.llm(self.CRITIC,
                         f"Task: {task}\nAnswer: {answer}\n\nEvaluation JSON:", max_tokens=150)
        try:
            raw  = re.sub(r"```(?:json)?\s*", "", resp.text).strip().rstrip("`")
            d    = json.loads(raw)
            return {"score": float(d.get("score",5)), "issues": d.get("issues",[]),
                    "should_retry": bool(d.get("should_retry", False))}
        except:
            return {"score": 5.0, "issues": ["parse error"], "should_retry": False}

    def run(self, task, verbose=True):
        history = []
        current_task = task
        for attempt in range(1, self.max_retries+2):
            if verbose: print(f"Attempt {attempt}:")
            r         = self.react.run(current_task)
            answer    = r["answer"]
            critique  = self._critique(task, answer)
            history.append({"attempt": attempt, "answer": answer, "critique": critique})
            if verbose:
                print(f"  Score: {critique['score']:.1f}/10  Issues: {critique['issues'][:2]}")
                print(f"  Answer: {answer[:90]}\n")
            if critique["score"] >= self.threshold or not critique["should_retry"]                or attempt > self.max_retries:
                break
            current_task = (f"{task}\n\n[Issues to fix: "
                            f"{', '.join(str(i) for i in critique['issues'][:2])}]")

        best = max(history, key=lambda x: x["critique"]["score"])
        return {"task": task, "best_answer": best["answer"],
                "best_score": best["critique"]["score"], "attempts": len(history)}


refl = ReflexionAgent(llm, TOOLS, max_retries=2, threshold=8.0)
r    = refl.run("Calculate compound interest on £10,000 at 5% for 3 years.", verbose=True)
print(f"Best answer (score={r['best_score']:.1f}, {r['attempts']} attempts):")
print(f"  {r['best_answer'][:150]}")


## 6. Budget Enforcement

In [ ]:
@dataclass
class AgentBudget:
    max_steps:   int   = 10
    max_tokens:  int   = 5_000
    max_time_s:  float = 60.0
    steps_used:  int   = 0
    tokens_used: int   = 0
    time_used_s: float = 0.0

    def check(self):
        if self.steps_used  >= self.max_steps:  return f"Step limit ({self.max_steps})"
        if self.tokens_used >= self.max_tokens: return f"Token limit ({self.max_tokens})"
        if self.time_used_s >= self.max_time_s: return f"Time limit ({self.max_time_s}s)"
        return None

    def update(self, tokens, elapsed_s):
        self.steps_used  += 1
        self.tokens_used += tokens
        self.time_used_s += elapsed_s

    def remaining(self):
        return {"steps": self.max_steps - self.steps_used,
                "tokens": self.max_tokens - self.tokens_used,
                "time_s": round(self.max_time_s - self.time_used_s, 1)}


class BudgetedAgent(ReActAgent):
    """ReAct agent that halts when any budget dimension is exceeded."""
    def __init__(self, llm, tools, budget):
        super().__init__(llm, tools, max_steps=budget.max_steps, verbose=True)
        self.budget = budget
        self._t0    = None

    def run(self, task):
        self._t0 = time.perf_counter()
        system   = self.SYSTEM.format(tool_list=self._tool_list())
        steps    = []
        for n in range(1, self.max_steps+1):
            limit = self.budget.check()
            if limit:
                print(f"  ⛔ Budget exceeded: {limit}")
                break
            user  = f"Task: {task}\n\n{self._trajectory(steps)}"
            t0    = time.perf_counter()
            resp  = self.llm(system, user, max_tokens=200)
            ms    = (time.perf_counter()-t0)*1000
            self.budget.update(resp.total_tokens, time.perf_counter()-self._t0)
            thought, action, args, is_final, answer = self._parse(resp.text)
            step = AgentStep(n, thought, action, args, is_final=is_final,
                              final_answer=answer, latency_ms=ms)
            if is_final: steps.append(step); self._print(step); break
            if action and action in self.tools:
                step.observation = json.dumps(call_tool(action, args or {}))[:200]
            steps.append(step); self._print(step)
        final = steps[-1].final_answer if steps and steps[-1].is_final else "Budget exceeded."
        return {"task": task, "answer": final, "steps": steps,
                "budget_used": {"steps": self.budget.steps_used,
                                "tokens": self.budget.tokens_used}}


budget = AgentBudget(max_steps=4, max_tokens=1500, max_time_s=30.0)
budgeted = BudgetedAgent(llm, TOOLS, budget)
result = budgeted.run("What is log(1000) and what is e raised to that power?")
print(f"\nBudget used: {result['budget_used']}")
print(f"Remaining:   {budget.remaining()}")


## 7. Agent Trajectory Visualisation

In [ ]:
# ── Run several tasks and visualise step counts ──────────────────────
agent_viz = ReActAgent(llm, TOOLS, max_steps=6, verbose=False)
tasks = [
    "Calculate the area of a circle with radius 5",
    "Search for FAISS documentation and summarise",
    "Write a Python function to compute factorial",
    "What is 15% of 847 plus sqrt(256)?",
]

step_counts, token_counts = [], []
for t in tasks:
    r = agent_viz.run(t)
    step_counts.append(r["n_steps"])
    token_counts.append(r["total_tokens"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("ReAct Agent: Trajectory Statistics", fontsize=11)
labels = [t[:30]+"..." for t in tasks]

ax1.barh(labels, step_counts, color="#3498db"); ax1.set_xlabel("Steps used")
ax1.set_title("Steps per task"); ax1.axvline(np.mean(step_counts), color="red",
               linestyle="--", alpha=0.7, label=f"mean={np.mean(step_counts):.1f}")
ax1.legend(); ax1.grid(axis="x", alpha=0.3)

ax2.scatter(step_counts, token_counts, s=120, c="#e67e22", zorder=3)
for i, t in enumerate(labels): ax2.annotate(t[:20], (step_counts[i], token_counts[i]),
                                              fontsize=7, xytext=(3,3), textcoords="offset points")
ax2.set_xlabel("Steps"); ax2.set_ylabel("Total tokens")
ax2.set_title("Tokens vs steps"); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f"Avg steps: {np.mean(step_counts):.1f}  Avg tokens: {np.mean(token_counts):.0f}")


## 8. Key Takeaways

| Concept | Summary |
|---|---|
| **ReAct** | Thought → Action → Observation loop; transparent reasoning |
| **Plan-and-Execute** | Decompose → execute subtasks → synthesise final answer |
| **Reflexion** | Self-critique score drives retries; converges on quality |
| **Budget** | Always cap max_steps, max_tokens, and max_time |
| **Trajectory** | Log every step for debugging and observability |
| **Human-in-the-loop** | Pause before irreversible write/communicate actions |

---
**Next notebook →** `04_agent_memory.ipynb`
